In [0]:
import requests
import json
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

# 1. Base External Location Path
bronze_base_path = "abfss://bronze@staecommercegen2.dfs.core.windows.net/"

# 2. Reliable E-Commerce Public API (DummyJSON)
def fetch_api_data(endpoint):
    url = f"https://dummyjson.com/{endpoint}"
    headers = {
        "User-Agent": "DatabricksPipeline/1.0"
    }
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        return data.get(endpoint, data)
    else:
        raise Exception(f"Failed to fetch {endpoint}: Status Code {response.status_code}")

# 3. Ingest Products, Users, and Carts Data (Serverless Friendly)
endpoints = ["products", "users", "carts"]

for ep in endpoints:
    print(f"Fetching {ep} from Public API...")
    raw_data = fetch_api_data(ep)
    
    # Step A: Convert JSON list to Pandas DataFrame
    pdf = pd.DataFrame(raw_data)
    
    # Step B: For nested columns/dicts, convert to string so PySpark can infer/load easily
    pdf = pdf.astype(str)
    
    # Step C: Convert Pandas DataFrame to PySpark DataFrame
    df_raw = spark.createDataFrame(pdf)
    
    # Step D: Add Bronze Metadata Columns
    df_bronze = df_raw.withColumn("ingestion_timestamp", current_timestamp()) \
                       .withColumn("source_system", lit("dummyjson_public_api"))
    #display(df_bronze)
    # Step E: Save as Delta Table in ADLS Gen2 Bronze Container
    output_path = f"{bronze_base_path}raw_{ep}"
    df_bronze.write.format("delta").mode("overwrite").save(output_path)
    print(f"✅ Successfully written raw_{ep} to {output_path}\n")

print("🚀 Public API Data Ingestion to Bronze Layer Complete!")

Fetching products from Public API...
✅ Successfully written raw_products to abfss://bronze@staecommercegen2.dfs.core.windows.net/raw_products

Fetching users from Public API...
✅ Successfully written raw_users to abfss://bronze@staecommercegen2.dfs.core.windows.net/raw_users

Fetching carts from Public API...
✅ Successfully written raw_carts to abfss://bronze@staecommercegen2.dfs.core.windows.net/raw_carts

🚀 Public API Data Ingestion to Bronze Layer Complete!
